# Review information: title + abstract without Main results and Authors' conclusions

Follow-up to [02_review_title_abstract.ipynb](02_review_title_abstract.ipynb) (FINDINGS.md, entry 2). There, the full Cochrane review abstract beat the LGAR criteria and research questions. Part of that abstract (*Main results*, *Authors' conclusions*) reports what the review found in its included studies, which a screener would not have at screening time.

**One factor changed relative to entry 2:** the review abstract drops *MAIN RESULTS* and *AUTHORS' CONCLUSIONS*. It keeps Background, Objectives, Search methods, Selection criteria and Data collection and analysis, which match what a review protocol has before screening. Records, record input, question and model are unchanged.

If the gain over the LGAR input holds, it does not come from post-screening text.

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = True    # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = True      # all 20 reviews; read the cost estimate first
DROP = {"MAIN RESULTS", "AUTHORS' CONCLUSIONS"}
OUT = S.out_dir("review_title_abstract_no_results")
COMPARE = {"baseline": S.out_dir("baseline"),
           "title_abstract": S.out_dir("review_title_abstract"),
           "no_results": OUT}

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
REVIEW_ABS = S.fetch_review_abstracts(CRITERIA)
SHORT = {r: S.drop_sections(a["abstract"], DROP) for r, a in REVIEW_ABS.items()}

n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")
pd.DataFrame({"full_chars": {r: len(a["abstract"]) for r, a in REVIEW_ABS.items()},
              "kept_chars": {r: len(s) for r, s in SHORT.items()},
              "kept_sections": {r: ", ".join(l.split(":")[0] for l in s.split("\n")) for r, s in SHORT.items()},
              "records": n, "in_subset": {r: r in SUBSET for r in n.index}})

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


,full_chars,kept_chars,kept_sections,records,in_subset
CD000996,3895,1694,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",281,True
CD001261,5834,1597,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",571,True
CD004414,6453,1592,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",336,True
CD006468,5497,1650,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",3874,False
CD007867,5124,1744,"BACKGROUND, OBJECTIVES, SELECTION CRITERIA, DA...",943,True
CD009069,6870,1484,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",1757,False
CD009642,6523,1985,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",1922,False
CD010038,6273,1968,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",8867,False
CD010239,4232,1652,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",224,True
CD010558,6558,2851,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT...",2815,False


## Decision model input

In [3]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {
            "review": {
                "title": CRITERIA[review]["title"],
                "abstract": SHORT[review],
            },
            "record": {"title": title, "abstract": abstract},
        },
        "questions": S.QUESTIONS,
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:3000])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "title": "Inhaled corticosteroids for bronchiectasis",
   "abstract": "BACKGROUND: Bronchiectasis is being increasingly diagnosed and recognised as an important contributor to chronic lung disease in both adults and children in high- and low-income countries. It is characterised by irreversible dilatation of airways and is generally associated with airway inflammation and chronic bacterial infection. Medical management largely aims to reduce morbidity by controlling the symptoms, reduce exacerbation frequency, improve quality of life and prevent the progression of bronchiectasis. This is an update of a review first published in 2000.\nOBJECTIVES: To evaluate the efficacy and safety of inhaled corticosteroids (ICS) in children and adults with stable state bronchiectasis, specifically to assess whether the use of ICS: (1) reduces the severity and frequency of acute respiratory exacerbations; or (2) affects long-term pulmonary 

## Cost estimate (no API call)

The 3.06 chars/token ratio overestimated the earlier runs by about 15%.

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")
est

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 4264 | max: 13715
estimated input tokens: 58.5M | estimated cost: $2.46
subset (10 reviews): $0.18


,records,chars,tokens_M,usd
review,,,,
CD000996,281,958632,0.313,0.013
CD001261,571,1898960,0.621,0.026
CD004414,336,1151185,0.376,0.016
CD006468,3874,13412477,4.383,0.184
CD007867,943,3679693,1.203,0.051
CD009069,1757,6187959,2.022,0.085
CD009642,1922,7799023,2.549,0.107
CD010038,8867,36768758,12.016,0.505
CD010239,224,877519,0.287,0.012


## Smoke test (one paid call)

In [5]:
if RUN_SMOKE:
    resp = S.call(build_body(*records.loc[0, ["review", "title", "abstract"]]))
    print(json.dumps(resp, indent=1))

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

reviews:   0%|          | 0/10 [00:00<?, ?it/s]

reviews: 100%|██████████| 10/10 [00:00<00:00, 251.00it/s]

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

reviews:   0%|          | 0/20 [00:00<?, ?it/s]

CD006468:   0%|          | 0/3874 [00:00<?, ?it/s]

CD006468:   0%|          | 1/3874 [00:00<22:28,  2.87it/s]

CD006468:   0%|          | 7/3874 [00:00<03:37, 17.81it/s]

CD006468:   0%|          | 10/3874 [00:00<03:29, 18.48it/s]

CD006468:   0%|          | 16/3874 [00:00<02:48, 22.88it/s]

... (8,834 outputs left out)

CD012455: 100%|█████████▉| 1586/1593 [00:53<00:00, 27.79it/s]

CD012455: 100%|█████████▉| 1590/1593 [00:53<00:00, 30.11it/s]

reviews:  95%|█████████▌| 19/20 [21:00<01:04, 64.95s/it]

reviews: 100%|██████████| 20/20 [21:00<00:00, 63.04s/it]

## Results

All configurations are scored on the same records: the reviews that have a cache file for this run. Failed calls and refusals get the lowest score and are counted. The other configurations are read from their caches, so nothing is re-requested.

In [8]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in COMPARE.items():
    sub = S.add_scores(sub, path, col)

results = {c: S.evaluate(sub, c) for c in ("bm25", *COMPARE)}
for c in ("title_abstract", "no_results"):
    results[f"{c} - baseline"] = results[c] - results["baseline"]
    results[f"{c} - baseline"]["reviews"] = results["baseline"]["reviews"]
pd.concat(results)

20 of 20 reviews run


baseline: scored 41996 of 41996 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


title_abstract: scored 41996 of 41996 records | model: {'typesafe/jev-1.14-20261014'} | cost: $3.61 | without a score (ranked last): 0
no_results: scored 41996 of 41996 records | model: {'typesafe/jev-1.14-20261014'} | cost: $2.06 | without a score (ranked last): 0


MAP  R@1%  R@5%  R@10%  R@20%  \
bm25                      label_abs      22.3  12.2  34.4   46.2   60.7   
                          label_content  17.0  12.0  30.2   47.0   65.2   
baseline                  label_abs      57.9  31.5  65.0   77.8   86.8   
                          label_content  60.7  45.3  84.2   90.9   94.9   
title_abstract            label_abs      61.5  32.7  65.4   78.9   88.8   
                          label_content  71.3  48.9  84.9   92.8   96.2   
no_results                label_abs      61.2  34.2  64.8   78.3   88.1   
                          label_content  70.6  50.9  85.9   92.3   95.7   
title_abstract - baseline label_abs       3.6   1.2   0.4    1.1    2.0   
                          label_content  10.6   3.6   0.7    1.9    1.3   
no_results - baseline     label_abs       3.3   2.7  -0.2    0.5    1.3   
                          label_content   9.9   5.6   1.7    1.4    0.8   

                                         R@50%  WSS@95%  TNR@95%  WSS@100%  \
bm25                      label_abs       82.6     33.3     39.5      28.7   
                          label_content   86.7     43.6     49.5      44.6   
baseline                  label_abs       97.0     66.3     74.7      62.1   
                          label_content   98.5     81.0     88.3      81.4   
title_abstract            label_abs       98.4     70.1     78.8      66.3   
                          label_content   99.5     83.5     91.1      85.2   
no_results                label_abs       99.0     68.1     76.8      64.8   
                          label_content   99.2     82.9     90.4      83.5   
title_abstract - baseline label_abs        1.4      3.8      4.1       4.2   
                          label_content    1.0      2.5      2.8       3.8   
no_results - baseline     label_abs        2.0      1.8      2.1       2.7   
                          label_content    0.7      1.9      2.1       2.1   

                                         reviews  
bm25                      label_abs         20.0  
                          label_content     18.0  
baseline                  label_abs         20.0  
                          label_content     18.0  
title_abstract            label_abs         20.0  
                          label_content     18.0  
no_results                label_abs         20.0  
                          label_content     18.0  
title_abstract - baseline label_abs         20.0  
                          label_content     18.0  
no_results - baseline     label_abs         20.0  
                          label_content     18.0

In [9]:
# per-review MAP
rows = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in COMPARE:
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            rows.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per_review = pd.DataFrame(rows).T
for label in ("label_abs", "label_content") if rows else ():
    d = per_review[(label, "no_results")] - per_review[(label, "baseline")]
    print(f"{label}: no_results vs baseline, better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
per_review

label_abs: no_results vs baseline, better 13 / worse 7 of 20 reviews
label_content: no_results vs baseline, better 15 / worse 3 of 18 reviews


label_abs                           label_content                 \
          baseline title_abstract no_results      baseline title_abstract   
CD000996      75.5           83.7       84.3          87.7          100.0   
CD001261      56.1           79.0       82.4          72.6           82.7   
CD004414      78.8           80.5       79.4          79.9           58.7   
CD006468      20.7           33.0       34.9          44.5           92.5   
CD007867      52.3           57.8       58.9          63.4           75.3   
CD009069      20.3           23.4       22.4           6.1            9.6   
CD009642      87.4           91.7       89.0          81.1           91.4   
CD010038      34.5           33.0       33.3          56.8           52.5   
CD010239      94.2           94.3       92.9          99.1           99.1   
CD010558      38.9           39.8       40.3          24.8           45.6   
CD010753      53.5           60.4       50.8          48.6           55.7   
CD011140      77.5           41.3       56.0           NaN            NaN   
CD011571      77.1           82.6       82.2          80.0           95.8   
CD011768      50.5           52.9       55.2          42.6           42.9   
CD011977      68.3           86.8       84.0          60.0           78.9   
CD012069      32.6           39.2       39.4          34.5           41.4   
CD012164      62.9           68.6       60.4          75.6           91.7   
CD012342      26.8           18.0       16.8           NaN            NaN   
CD012455      86.9          100.0       98.2          58.7           92.7   
CD012551      64.1           63.6       63.2          76.7           77.2   

                     
         no_results  
CD000996      100.0  
CD001261       80.9  
CD004414       73.7  
CD006468       93.5  
CD007867       79.1  
CD009069        8.6  
CD009642       88.4  
CD010038       54.7  
CD010239      100.0  
CD010558       33.3  
CD010753       48.2  
CD011140        NaN  
CD011571       97.6  
CD011768       46.0  
CD011977       78.2  
CD012069       41.7  
CD012164       80.6  
CD012342        NaN  
CD012455       87.7  
CD012551       78.8